# 🧬 BraveNewAlgorithm.jl — BBOB Sphere Function Demo

## ⚡ Google Colab Quick-Start

Google Colab does not include Julia by default. Follow these steps **once per
session** to get Julia running:

1. **Run the next cell** (click it, then press **Ctrl+Enter** or click ▶).  
   It downloads Julia 1.11, installs the required Julia packages, and registers
   the Julia kernel with Colab.  This takes 3–5 minutes.
2. **Reload the page** (press **Ctrl+R** / **⌘+R** / **F5**).  
   After the reload Colab will automatically use the Julia kernel for all
   subsequent cells.
3. **Continue from section 1** — skip the Colab install cell.

> *If your Colab Runtime is reset (e.g., due to inactivity), repeat steps 1–2.*

---

**Running locally with Jupyter?**  
Skip the install cell entirely — jump straight to **Section 1 · Package Setup**.


In [ ]:
%%shell
# ── Colab-only cell: installs Julia 1.11 and required packages ───────────────
# Skip this cell when running locally in Jupyter (Julia is already installed).
set -e

#--------------------------------------------------------------------#
JULIA_VERSION="1.11.4"          # Julia release to install
JULIA_NUM_THREADS=2              # parallel threads
#--------------------------------------------------------------------#

if [ -z "$(which julia 2>/dev/null)" ]; then
  JULIA_MAJ_MIN=$(echo "$JULIA_VERSION" | cut -d '.' -f1-2)
  TARBALL="julia-${JULIA_VERSION}-linux-x86_64.tar.gz"
  URL="https://julialang-s3.julialang.org/bin/linux/x64/${JULIA_MAJ_MIN}/${TARBALL}"

  echo "Downloading Julia ${JULIA_VERSION}..."
  wget -q "$URL" -O /tmp/julia.tar.gz
  tar -x -f /tmp/julia.tar.gz -C /usr/local --strip-components 1
  rm /tmp/julia.tar.gz
  echo "Julia installed: $(julia -v)"

  echo "Installing IJulia and DataFrames..."
  julia -e 'using Pkg; Pkg.add(["IJulia", "DataFrames"]); Pkg.precompile()' &>/dev/null

  echo "Installing BraveNewAlgorithm.jl from GitHub..."
  julia -e 'using Pkg; Pkg.add(url="https://github.com/cecimerelo/BraveNewAlgorithm.jl"); Pkg.precompile()' &>/dev/null

  echo "Installing BlackBoxOptimizationBenchmarking v1.x (required API)..."
  julia -e 'using Pkg; Pkg.add(PackageSpec(name="BlackBoxOptimizationBenchmarking", version="1")); Pkg.precompile()' &>/dev/null

  echo "Registering Julia kernel..."
  julia -e 'using IJulia; IJulia.installkernel("Julia 1.11", env=Dict("JULIA_NUM_THREADS"=>"'"$JULIA_NUM_THREADS"'"))'

  echo ""
  echo "✅ $(julia -v) installed with all required packages."
  echo "👉 Please RELOAD this page (Ctrl+R / ⌘+R / F5), then skip this cell"
  echo "   and continue from Section 1."
else
  echo "Julia already installed: $(julia -v) — nothing to do."
fi


## About This Notebook

**BraveNewAlgorithm.jl** is a metaheuristic optimisation algorithm inspired by
Aldous Huxley's *Brave New World*.
The population is divided into five **castes** — mirroring the novel's social
hierarchy — each playing a different role in balancing *exploration*
(discovering new regions of the search space) and *exploitation*
(refining already-promising solutions).

This notebook uses a **Julia kernel** so every code cell runs Julia directly.
It walks you through:

1. Loading the package and its dependencies
2. Understanding the five-caste system and its parameters
3. Running the BBOB Sphere optimisation
4. Inspecting convergence results
5. Comparing three caste-distribution strategies


## 1. Package Setup

**Two ways to run this notebook:**

### Option A — cloned repository (default)
The cell below activates the local project environment and calls
`Pkg.instantiate()` to install every dependency declared in `Project.toml`
(including `BlackBoxOptimizationBenchmarking < 2`).

### Option B — standalone / independent environment
Comment out the Option-A lines and uncomment the Option-B block to
install all required packages from scratch into a fresh environment.


In [ ]:
using Pkg

# ── Option A: running from the cloned repository ─────────────────────────────
# Activates the project environment and installs all pinned dependencies.
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()  # installs all deps from Project.toml (incl. BlackBoxOptimizationBenchmarking <2)

# ── Option B: standalone / independent environment ───────────────────────────
# Comment out Option A above, then uncomment the lines below.
# Pkg.add(url = "https://github.com/cecimerelo/BraveNewAlgorithm.jl")
# Pkg.add(PackageSpec(name="BlackBoxOptimizationBenchmarking", version="1"))  # must be v1.x (<2)
# Pkg.add("DataFrames")

using BraveNewAlgorithm
using BlackBoxOptimizationBenchmarking
using DataFrames

println("✅ Packages loaded")

## 2. The Brave New World Caste System

Just as Huxley's society assigns each person a predetermined role, the algorithm
assigns each individual to a caste that governs how it is selected, crossed over,
and mutated:

| Caste   | Role in optimisation            | Mutation (example) |
|---------|---------------------------------|--------------------|
| ALPHA   | Elite — best solutions kept     | Low (e.g. 20 %)    |
| BETA    | High performers — local search  | Low (e.g. 20 %)    |
| GAMMA   | Average — balanced search       | Medium (e.g. 40 %) |
| DELTA   | Below average — diversify       | Medium (e.g. 40 %) |
| EPSILON | Lowest fitness — pure explore   | High (e.g. 60 %)   |

> *The mutation values shown above are for the **5-dimensional** example in this
> notebook.  For a chromosome of size `n`, the minimum effective rate is
> `ceil(100/n) %` (lower rates round down to 0 gene changes).*

**Constructor constraints** enforced automatically:

- `BETA_pct = 2 × ALPHA_pct` (balanced elite reproduction)
- All five percentages must sum to **100**
- `ALPHA_count = ALPHA_pct × population_size / 100` must be an even integer
  (and likewise for BETA)

### Algorithm flow

```
Initialise population (fertilising room)
         │
         ▼
 ┌─── Assign castes (hatchery) ◄──────────────────┐
 │         │                                       │
 │         ▼                                       │
 │   Apply genetic operators per caste             │
 │   (selection → crossover → mutation)            │
 │         │                                       │
 │         ▼                                       │
 │   Elitism: inject best from previous generation │
 │         │                                       │
 │         ▼                                       │
 │   Stopping criterion met?  ──No──────────────────┘
 │         │ Yes
 └─────────▼
       Return results
```


## 3. The BBOB Sphere Function

BBOB function #1 is the **Sphere** function — the simplest unimodal benchmark:

$$f(\mathbf{x}) = \sum_{i=1}^{n} (x_i - x_i^*)^2 + f_{\text{opt}}$$

where $x^*$ is a random shift (hidden inside the BBOB instance) and
$f_{\text{opt}}$ is the true minimum value.

BraveNewAlgorithm **maximises** the negated fitness, so the stopping criterion
is met when `best_f_value ≥ f_opt + ε`.


## 4. Configuring the Algorithm

`ConfigurationParametersEntity` accepts:

| Parameter              | Description                                      |
|------------------------|--------------------------------------------------|
| `chromosome_size`      | Number of problem dimensions                     |
| `population_size`      | Total number of individuals                      |
| `max_generations`      | Max consecutive generations without improvement  |
| `max_hillclimbing_steps` | Local-search steps per individual              |
| `castes_percentages`   | Dict mapping caste name → % of population        |
| `mutation_rate`        | Dict mapping caste name → % of genes to mutate  |

> **Mutation rate constraint**: the mutation rate is the *percentage of genes*
> that change, rounded down (`floor(rate/100 × chromosome_size)`).  
> For a 5-dimensional chromosome only multiples of 20 % produce a non-zero
> number of mutations (20 % → 1 gene, 40 % → 2, 60 % → 3, …).  
> In general, the minimum effective rate is `ceil(100 / chromosome_size)` %.


In [ ]:
# Population of 100 makes caste counts exact integers and easy to reason about.
# For chromosome_size=5 the mutation rate must be a multiple of 20%
# (floor(rate/100 × 5) must be ≥ 1, so the minimum effective rate is 20%).

config = ConfigurationParametersEntity(
    5,       # chromosome_size  — 5-dimensional Sphere
    100,     # population_size  — 100 individuals
    50,      # max_generations  — stop after 50 stagnant generations
    10,      # max_hillclimbing_steps
    Dict{String, Int}(
        "ALPHA"   => 10,   # 10 individuals — elite
        "BETA"    => 20,   # 20 individuals — high performers  (= 2 × ALPHA ✓)
        "GAMMA"   => 30,   # 30 individuals — average
        "DELTA"   => 25,   # 25 individuals — below average
        "EPSILON" => 15,   # 15 individuals — exploratory
    ),
    Dict{String, Int}(
        "ALPHA"   => 20,   # 20% → 1/5 genes mutate (minimum for 5-dim)
        "BETA"    => 20,   # 20% → 1/5 genes mutate
        "GAMMA"   => 40,   # 40% → 2/5 genes mutate
        "DELTA"   => 40,   # 40% → 2/5 genes mutate
        "EPSILON" => 60,   # 60% → 3/5 genes mutate — maximum diversity
    )
)

println("Configuration created:")
println("  Dimensions         : ", config.chromosome_size)
println("  Population size    : ", config.population_size)
println("  Max stagnation     : ", config.max_generations, " generations")
println("  Caste distribution : ", config.castes_percentages)

## 5. Fitness Function and Population Model

`FitnessFunction` wraps a BBOB function and counts evaluations.
The comparator receives `(best_f_value, bbob_function)` — note that the second
argument is the raw `BBOBFunction` (not the `FitnessFunction` wrapper), so use
`bbob_ff.f_opt` to access the optimum.

> **Package version note**: `BlackBoxOptimizationBenchmarking.BBOBFunctions`
> is the API provided by version **1.x** of the package.  
> Version 2+ changed the API; this notebook requires version **< 2**,
> which is enforced by `Project.toml` (Option A) or by the
> `PackageSpec(name=..., version="1")` `Pkg.add` call (Option B).


In [ ]:
# Sphere function (BBOB #1)
ff = FitnessFunction(BlackBoxOptimizationBenchmarking.BBOBFunctions[1])

println("Fitness function : BBOB Sphere (Function #1)")
println("Optimal value f* : ", ff.fitness_function.f_opt)

search_range = (-5.0, 5.0)

# Stop when best value is within 1e-6 of the optimum.
# The second argument is the raw BBOBFunction, not the FitnessFunction wrapper.
comparator = (best_val, bbob_ff) -> best_val >= bbob_ff.f_opt + 1e-6

model = PopulationModel(config, ff, search_range, comparator)
println("Population model ready ✅")

## 6. Running the Algorithm

`brave_new_algorithm(model)` returns `(generation, results)` where:

- `generation` — total generations executed
- `results` — a `DataFrame` with columns `Generations` (Int) and
  `F_Values` (Float64, best fitness per generation)

The loop terminates when either:

- the comparator is satisfied (optimum reached within tolerance), **or**
- the best individual has not improved for `max_generations` consecutive
  generations.


In [ ]:
println("Starting optimisation…")
start_time = time()

generation, results = brave_new_algorithm(model)

elapsed = round(time() - start_time; digits=2)
println("Done in $(elapsed) s")

## 7. Results


In [ ]:
best_fitness = minimum(results.F_Values)
f_opt        = ff.fitness_function.f_opt
gap          = best_fitness - f_opt

println("Generations run      : ", generation)
println("Function evaluations : ", ff.calls_counter)
println("Best fitness found   : ", round(best_fitness; digits=8))
println("Optimal value (f*)   : ", round(f_opt;        digits=8))
println("Gap to optimum       : ", gap < 1e-6 ? "< 1e-6  ✅ (optimum reached)" : string(round(gap; digits=6)))

### Convergence History (last 10 recorded generations)


In [ ]:
n = length(results.F_Values)
start_idx = max(1, n - 9)

println(" Generation | Best Fitness")
println("------------|------------------")
for i in start_idx:n
    gen  = lpad(string(results.Generations[i]), 10)
    fval = rpad(string(round(results.F_Values[i]; digits=6)), 18)
    println(" ", gen, " | ", fval)
end

### Convergence Plot (ASCII)

A simple log-scale ASCII chart of best fitness vs. generation.
Install `UnicodePlots.jl` (`Pkg.add("UnicodePlots")`) for a nicer in-terminal
plot, or use `Plots.jl` for publication-quality output.


In [ ]:
# ── Simple ASCII convergence chart (no extra packages needed) ──
using Printf

fitness_vals = results.F_Values
gen_nums = results.Generations

if length(fitness_vals) >= 2
    vmin = minimum(fitness_vals)
    vmax = maximum(fitness_vals)
    width = 50  # chart width in characters

    println("\nBest fitness over generations (scaled to chart width)")
    println("f_max=$(round(vmax;digits=4))  f_min=$(round(vmin;digits=4))")
    println("  ", "-" ^ (width + 4))

    for i in 1:length(fitness_vals)
        # Map value linearly to bar length (higher fitness = longer bar)
        range_v = vmax - vmin
        if range_v ≈ 0
            println("All fitness values are identical — no convergence to plot.")
            break
        end
        bar_len = clamp(round(Int, (fitness_vals[i] - vmin) / range_v * width), 0, width)
        @printf "%4d | %s\n" gen_nums[i] ("█" ^ bar_len)
    end
    println("  ", "-" ^ (width + 4))
else
    println("Not enough data points to plot.")
end

## 8. Caste-Distribution Strategy Comparison

Different allocations of individuals across castes shift the exploration–exploitation
balance.  We test three strategies on the same 5-D Sphere problem:

| Strategy           | ALPHA | BETA | GAMMA | DELTA | EPSILON | Focus       |
|--------------------|-------|------|-------|-------|---------|-------------|
| Exploitation-heavy | 20    | 40   | 20    | 10    | 10      | Refining    |
| Balanced           | 10    | 20   | 30    | 25    | 15      | General     |
| Exploration-heavy  | 10    | 20   | 20    | 20    | 30      | Diversity   |

All use `population_size = 100` so counts are exact integers, and all satisfy
`BETA = 2 × ALPHA`.


In [ ]:
strategies = [
    ("Exploitation-heavy",
        Dict("ALPHA"=>20,"BETA"=>40,"GAMMA"=>20,"DELTA"=>10,"EPSILON"=>10),
        Dict("ALPHA"=>20,"BETA"=>20,"GAMMA"=>20,"DELTA"=>40,"EPSILON"=>40)),
    ("Balanced",
        Dict("ALPHA"=>10,"BETA"=>20,"GAMMA"=>30,"DELTA"=>25,"EPSILON"=>15),
        Dict("ALPHA"=>20,"BETA"=>20,"GAMMA"=>40,"DELTA"=>40,"EPSILON"=>60)),
    ("Exploration-heavy",
        Dict("ALPHA"=>10,"BETA"=>20,"GAMMA"=>20,"DELTA"=>20,"EPSILON"=>30),
        Dict("ALPHA"=>20,"BETA"=>20,"GAMMA"=>40,"DELTA"=>40,"EPSILON"=>80)),
]

println(rpad("Strategy", 20), " | ", lpad("Gap to f*", 14), " | ", lpad("Generations", 11))
println("-" ^ 53)

for (name, castes, mutations) in strategies
    cfg  = ConfigurationParametersEntity(5, 100, 50, 10,
               Dict{String,Int}(castes), Dict{String,Int}(mutations))
    fit  = FitnessFunction(BlackBoxOptimizationBenchmarking.BBOBFunctions[1])
    cmp  = (val, bbob_ff) -> val >= bbob_ff.f_opt + 1e-6
    mdl  = PopulationModel(cfg, fit, (-5.0, 5.0), cmp)
    gen, res = brave_new_algorithm(mdl)
    gap  = minimum(res.F_Values) - fit.fitness_function.f_opt
    gap_str = gap < 1e-6 ? "< 1e-6 ✅" : string(round(gap; digits=4))
    println(rpad(name, 20), " | ", lpad(gap_str, 14), " | ", lpad(string(gen), 11))
end

## 9. Customisation Guide

### Changing the problem

```julia
# Use any of the 24 BBOB functions (requires BlackBoxOptimizationBenchmarking v1.x):
ff = FitnessFunction(BlackBoxOptimizationBenchmarking.BBOBFunctions[2])  # Rosenbrock
ff = FitnessFunction(BlackBoxOptimizationBenchmarking.BBOBFunctions[3])  # Ellipsoid
```

### Changing dimensions

For a 10-dimensional problem the minimum effective mutation rate is
`ceil(100/10) = 10 %`.  Use multiples of 10 % to ensure at least one gene changes:

```julia
config = ConfigurationParametersEntity(
    10, 100, 100, 20,
    Dict("ALPHA"=>10,"BETA"=>20,"GAMMA"=>30,"DELTA"=>25,"EPSILON"=>15),
    Dict("ALPHA"=>10,"BETA"=>10,"GAMMA"=>20,"DELTA"=>20,"EPSILON"=>30),
)
```

### Tighter stopping criterion

```julia
comparator = (val, bbob_ff) -> val >= bbob_ff.f_opt + 1e-8
```

### Reading configuration from a JSON file

```julia
using JSON
config = read_parameters_file("path/to/config.json")
```

A minimal JSON configuration file looks like:

```json
{
  "CHROMOSOME_SIZE": 5,
  "POPULATION_SIZE": 100,
  "MAX_GENERATIONS": 50,
  "MAX_HILLCLIMBING_STEPS": 10,
  "POPULATION_PERCENTAGE": { "ALPHA":10,"BETA":20,"GAMMA":30,"DELTA":25,"EPSILON":15 },
  "MUTATION_RATE":         { "ALPHA":20,"BETA":20,"GAMMA":40,"DELTA":40,"EPSILON":60 }
}
```


## Citation

If you use BraveNewAlgorithm.jl in your research, please cite:

```bibtex
@misc{merelo2023bravenew,
  author = {Merelo, J.J. and Merelo-Molina, Cecilia},
  title  = {{BraveNewAlgorithm.jl}: A caste-based metaheuristic in Julia},
  year   = {2023},
  url    = {https://github.com/cecimerelo/BraveNewAlgorithm.jl}
}
```

---
*Inspired by Aldous Huxley's*
[*Brave New World*](https://en.wikipedia.org/wiki/Brave_New_World) *(1932).*
